# 03 · Models and training

This notebook prepares the ten models (Supplementary Table 1).

1. **Pretrained models and third-party code**: where to obtain the weights of the Lima et al. model, ECGFounder, ECG-FM and
   HuBERT-ECG and the code of their developers; the checksums of the weight files are compared with the expected values, and
   every network is loaded once.
2. **Frozen embeddings** of every MIMIC-IV-ECG record for the three foundation models.
3. **Frozen versions**: ridge regression for age and logistic regression for sex on the embeddings of the training split.
4. **Fine-tuned versions**: each foundation model, with linear heads for age and sex, trained on the training split of
   MIMIC-IV-ECG and, separately, of CODE-15%.

The Lima et al. model is used with its published weights and is not trained. Notebook 04 applies the ten models to the ECGs of the test split.

| Output | Content |
|---|---|
| `EMBEDDINGS/{ecgfounder, ecgfm, hubert}_f16.npy` | (800,035 × 1,024 / 768 / 960) float16; row = idx |
| `TRAINED/{key}/ridge.npz` | the fitted regressions of the three frozen models (content in section 3) |
| `TRAINED/{key}/best.pt` | the checkpoint with the lowest validation MAE of each of the six fine-tuned models |
| `TRAINING/{key}.json` | settings, the chosen penalty or checkpoint, and the validation history |

**Needs**: notebooks 01 and 02 (waveform arrays, HuBERT-ECG inputs, cohort with splits, ages and sexes); one CUDA graphics card
with about 10 GB of free memory for sections 2 and 4; about 12 GB of RAM for section 3.

**Run time** on one NVIDIA RTX 5090: embeddings about 10 min per foundation model; probes 1–5 min each on the CPU; fine-tuning,
including the validations, about 20 min (ECGFounder), 50–60 min (ECG-FM) and 110 min (HuBERT-ECG) per dataset. Every step skips
its output when it exists, and the fine-tuning resumes from its last completed epoch.

**Credentialed data.** The frozen models and the models fine-tuned on MIMIC-IV-ECG are trained with ages and sexes from MIMIC-IV:
their files stay in `MASTER` and must not be shared. Only aggregate numbers are printed.

In [ ]:
%run ./00_setup_and_settings.ipynb

## What to run

`QUICK_TEST = True` runs every step on a tiny subset (a few minutes in all) and writes only below `QUICK_DIR`, never to the paths
of the full run: embeddings of the first 2,048 records, probes on these records, and fine-tuning for three short epochs on the
first 1,024 training and 256 validation records, with a validation every 5 steps.

In [ ]:
# ============================== WHAT TO RUN ==============================
QUICK_TEST = False
QUICK_DIR = MASTER / "_quick"                      # outputs of the quick test
DOWNLOAD = False                                   # True: download the pretrained weights and clone the developers' code (section 1)
RUN_EMBEDDINGS = ["ecgfounder", "ecgfm", "hubert"]
RUN_PROBES = ["ecgfounder_frozen", "ecgfm_frozen", "hubert_frozen"]
RUN_FINETUNE = ["ecgfounder_mimic", "ecgfm_mimic", "hubert_mimic", "ecgfounder_code15", "ecgfm_code15", "hubert_code15"]
QUICK_EMBED_ROWS = 2048
QUICK_TRAIN_ROWS = 1024
QUICK_VAL_ROWS = 256

EMB_DIR = QUICK_DIR / "embeddings" if QUICK_TEST else EMBEDDINGS
TRAINED_DIR = QUICK_DIR / "trained" if QUICK_TEST else TRAINED
TRAINING_DIR = QUICK_DIR / "training" if QUICK_TEST else TRAINING
for _d in (EMB_DIR, TRAINED_DIR, TRAINING_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print("quick test" if QUICK_TEST else "full run", "| embeddings ->", EMB_DIR, "| models ->", TRAINED_DIR, "| records ->", TRAINING_DIR)

## 1. Pretrained models and third-party code

### 1.1 Weights

The weights are not part of this repository. Download them from their developers at the revisions listed (the cell below does it
when `DOWNLOAD = True`) and place them as shown; section 1.3 compares their checksums with the expected values.

| Model | Files | Source and revision | Licence | Place under `PRETRAINED` |
|---|---|---|---|---|
| Lima et al. | `model.zip` (63,003,529 bytes, MD5 `00aad2c0341942837019086cf5d94cd5`), containing `model/config.json`, `model/model.pth` | Zenodo record 4892365, version 1.0.0 ([doi:10.5281/zenodo.4892365](https://doi.org/10.5281/zenodo.4892365)) | CC BY 4.0 | unzip into `lima2021/weights/` |
| ECGFounder | `12_lead_ECGFounder.pth` | Hugging Face `PKUDigitalHealth/ECGFounder`, revision `d9b1793951b2342f5f7e84f1ac03cd37f8a08724` | MIT (model card) | `ecgfounder/` |
| ECG-FM | `mimic_iv_ecg_physionet_pretrained.pt` | Hugging Face `wanglab/ecg-fm`, revision `584219ea492cdeef2e19ffbdf9c6ecc874ba427e` | MIT | `ecg-fm/` |
| HuBERT-ECG large | `config.json`, `model.safetensors` | Hugging Face `Edoardo-Coppola/hubert-ecg-large`, revision `f297337d4e33198800441e3d64e0dd393774637f` | CC BY-NC 4.0 (non-commercial use only) | `hubert-ecg/large/` |

### 1.2 Code of the developers

The code is not copied into this repository. Clone the repositories into `THIRD_PARTY` and check out the commits listed (the cell
below does it when `DOWNLOAD = True`).

| Repository | Commit | Licence | Clone into `THIRD_PARTY` | Used for |
|---|---|---|---|---|
| https://github.com/antonior92/ecg-age-prediction | `f192e49e095708c038e4f430a41ad31fc4d3f8bd` | MIT | `ecg-age-prediction/` | `resnet.py`: the network of the Lima et al. model (imported) |
| https://github.com/PKUDigitalHealth/ECGFounder | `68d25f25e323a4a423b9d9e8ea2e0af3f234bf22` | no licence file in the repository; the model card states MIT | `ECGFounder/` | `net1d.py`: the network of ECGFounder (imported) |
| https://github.com/Jwoo5/fairseq-signals | `f8f0ff1c788a82c2059cb452cd5462898867489e` | MIT | `fairseq-signals/` | reference for the ECG-FM encoder, which notebook 00 writes out in plain PyTorch with the parameter names of the released checkpoint (not imported) |
| https://github.com/bowang-lab/ECG-FM | `9f926f1911bb9f24789b5c6407677d58ad753054` | MIT | `ECG-FM/` | reference for the ECG-FM inputs and the embedding (mean over time of the last layer) |
| https://github.com/Edoar-do/HuBERT-ECG | `36061fc5a8bc0d3adaf9df9dd8ef551c6391a6e9` | CC BY-NC 4.0 | `HuBERT-ECG/` | reference for the HuBERT-ECG input preprocessing (notebook 01); the network is the standard HuBERT of `transformers` |

In [ ]:
HF_WEIGHTS = [   # (repository, revision, files, folder)
    ("PKUDigitalHealth/ECGFounder", "d9b1793951b2342f5f7e84f1ac03cd37f8a08724", ["12_lead_ECGFounder.pth"], ECGFOUNDER_WEIGHTS.parent),
    ("wanglab/ecg-fm", "584219ea492cdeef2e19ffbdf9c6ecc874ba427e", ["mimic_iv_ecg_physionet_pretrained.pt"], ECGFM_WEIGHTS.parent),
    ("Edoardo-Coppola/hubert-ecg-large", "f297337d4e33198800441e3d64e0dd393774637f", ["config.json", "model.safetensors"], HUBERT_WEIGHTS),
]
LIMA_ZIP_URL = "https://zenodo.org/records/4892365/files/model.zip?download=1"
CODE_REPOS = [   # (repository, commit, folder)
    ("https://github.com/antonior92/ecg-age-prediction", "f192e49e095708c038e4f430a41ad31fc4d3f8bd", LIMA_CODE),
    ("https://github.com/PKUDigitalHealth/ECGFounder", "68d25f25e323a4a423b9d9e8ea2e0af3f234bf22", ECGFOUNDER_CODE),
    ("https://github.com/Jwoo5/fairseq-signals", "f8f0ff1c788a82c2059cb452cd5462898867489e", THIRD_PARTY / "fairseq-signals"),
    ("https://github.com/bowang-lab/ECG-FM", "9f926f1911bb9f24789b5c6407677d58ad753054", THIRD_PARTY / "ECG-FM"),
    ("https://github.com/Edoar-do/HuBERT-ECG", "36061fc5a8bc0d3adaf9df9dd8ef551c6391a6e9", THIRD_PARTY / "HuBERT-ECG"),
]

if DOWNLOAD:
    import subprocess
    import urllib.request
    import zipfile
    from huggingface_hub import hf_hub_download
    for repo, rev, files, folder in HF_WEIGHTS:
        folder.mkdir(parents=True, exist_ok=True)
        for f in files:
            if not (folder / f).exists():
                hf_hub_download(repo, f, revision=rev, local_dir=folder)
    zpath = PRETRAINED / "lima2021" / "model.zip"
    if not (LIMA_WEIGHTS / "model.pth").exists():
        zpath.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(LIMA_ZIP_URL, zpath)
        with zipfile.ZipFile(zpath) as z:
            assert not [m for m in z.namelist() if m.startswith(("/", "\\")) or ".." in Path(m).parts], "unsafe path in model.zip"
            z.extractall(PRETRAINED / "lima2021" / "weights")
    for url, commit, folder in CODE_REPOS:
        if not folder.exists():
            subprocess.run(["git", "clone", url, str(folder)], check=True)
            subprocess.run(["git", "-C", str(folder), "checkout", commit], check=True)

### 1.3 Check the files

Expected SHA256 checksums of the weight files. The cell prints OK, missing or CHECKSUM DIFFERS for each file, and whether
`resnet.py` and `net1d.py` of the developers' code are present.

In [ ]:
import hashlib

WEIGHT_SHA256 = {
    LIMA_WEIGHTS / "model.pth": "6d649c0323b36fa0d7a33119cd61a049d2bbc24f860205ec8bc48098726ef40b",
    LIMA_WEIGHTS / "config.json": "b09ae3f4410cc833a5994b01b9388e3bc3d1fc43ae627d449e392a4f34c53fe4",
    ECGFOUNDER_WEIGHTS: "ee199f3781f4ae1f732973267f003da0a759ea12bddb0dd28a77faa60aca7997",
    ECGFM_WEIGHTS: "4d0142bcb485eb9f0c7845e0c19ff3463f6ae9d0e458eab69136efe90ceb9b7e",
    HUBERT_WEIGHTS / "config.json": "6b28e6a2321cf0ec341bb8e51b8d630d8de21bce0af218ff27fd99c12f2cbf6b",
    HUBERT_WEIGHTS / "model.safetensors": "9a0eb8484f73793aa678936984243da92ae80d7b81e2fe77d1ea6dee2bdf5ec2",
}


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(1 << 24):
            h.update(chunk)
    return h.hexdigest()


for path, expected in WEIGHT_SHA256.items():        # about 15 s
    status = "missing" if not path.exists() else ("OK" if sha256(path) == expected else "CHECKSUM DIFFERS")
    print(f"{status:16s} {path.relative_to(PRETRAINED)}")
for path in (LIMA_CODE / "resnet.py", ECGFOUNDER_CODE / "net1d.py"):
    print(f"{'OK' if path.exists() else 'missing':16s} {path.relative_to(THIRD_PARTY)}")

### 1.4 Load every network once

Each network is built and its pretrained weights are loaded (all with `strict=True`, except HuBERT-ECG, whose release lacks only
the unused mask embedding). The cell prints the number of parameters of each network (Supplementary Table 1), and a random batch
checks the output shapes.

In [ ]:
import torch

with torch.no_grad():                               # on the CPU, about 1 min
    lima = lima_model()
    print(f"Lima et al.: {sum(p.numel() for p in lima.parameters()) / 1e6:.1f} million parameters, output",
          tuple(lima(torch.randn(2, 12, 4096)).shape))
    del lima
    for fm, shape in (("ecgfounder", (2, 12, 5000)), ("ecgfm", (2, 12, 5000)), ("hubert", (2, 2, 6000))):
        net = age_sex_model(fm, pretrained=True).eval()
        x = torch.randint(-400, 400, shape, dtype=torch.int16) if fm != "hubert" else (torch.rand(shape) * 2 - 1).half()
        a, s, f = net(net.prep(x))
        print(f"{fm}: {sum(p.numel() for p in net.parameters()) / 1e6:.1f} million parameters (with the two heads), "
              f"embedding {tuple(f.shape)}, finite {bool(torch.isfinite(f).all())}")
        del net

## 2. Frozen embeddings

The embedding of every MIMIC-IV-ECG record (row = idx, also excluded records) is computed once with the pretrained network and
stored in float16. Numerical settings: ECGFounder in float32 (TensorFloat-32 allowed), ECG-FM and HuBERT-ECG in bfloat16 mixed
precision; 64 records per batch. The file is written under a temporary name and renamed when complete.

In [ ]:
import time


def compute_embeddings(fm, path, n_rows=None, batch_size=64):
    dev = gpu_device()
    if fm == "ecgfounder":
        gpu_precision(tf32_matmul=True, tf32_conv=True, benchmark=False)
    else:
        gpu_precision(tf32_matmul=False, tf32_conv=True, benchmark=True)
    arr = model_inputs("mimic", fm)
    n = arr.shape[0] if n_rows is None else min(n_rows, arr.shape[0])
    model = age_sex_model(fm, pretrained=True).to(dev).eval()
    tmp = path.with_name(path.stem + ".partial.npy")
    out = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(n, EMBED_DIM[fm]))
    t0, done = time.time(), 0
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=fm != "ecgfounder"):
        for rows, x in block_batches(arr, np.arange(n), batch_size, block=4096):
            f = model.embed(model.prep(torch.from_numpy(np.ascontiguousarray(x)).to(dev, non_blocking=True)))
            out[rows] = f.float().cpu().numpy().astype(np.float16)
            done += len(rows)
            if done % (batch_size * 2000) < batch_size:
                print(f"  {fm}: {done:,} of {n:,} records ({done / (time.time() - t0):.0f} per s)", flush=True)
    finite = bool(np.isfinite(np.asarray(out[:: max(1, n // 5000)], dtype=np.float32)).all())
    out.flush()
    del out, model
    torch.cuda.empty_cache()
    tmp.replace(path)
    print(f"{fm}: {n:,} x {EMBED_DIM[fm]} in {(time.time() - t0) / 60:.1f} min; finite (sample of rows): {finite}")


for fm in RUN_EMBEDDINGS:                           # about 10 min per foundation model on one RTX 5090
    path = EMB_DIR / f"{fm}_f16.npy"
    if path.exists():
        print(f"{fm}: embeddings exist, skipped")
        continue
    compute_embeddings(fm, path, n_rows=QUICK_EMBED_ROWS if QUICK_TEST else None)

## 3. Frozen versions: ridge and logistic regression on the embeddings

The embeddings of the MIMIC-IV-ECG training split are standardised with the mean and SD of each dimension in the training split
(SD + 10⁻⁶), and the same standardisation is applied to all records.

* **Age**: ridge regression on all training ECGs with a known age, with an unpenalised intercept (the mean training age), solved in
  closed form for the penalties 10⁻¹ … 10⁵; the penalty with the lowest validation MAE (each patient weighted equally) is kept.
* **Sex**: logistic regression (L2 penalty, L-BFGS, at most 500 iterations) fitted to a random subsample of 200,000 training ECGs
  with a known sex (generator seed 0), for the inverse penalties C = 10⁻³ … 1; the C with the highest validation AUC is kept.

`TRAINED/{key}/ridge.npz` holds everything needed to apply the probe (notebook 04):

| Array | Content |
|---|---|
| `feature_mean`, `feature_sd` | standardisation of the embedding (float32, one value per dimension; SD includes the + 10⁻⁶) |
| `age_coef`, `age_intercept` | ridge coefficients and intercept: age = ((embedding − mean) / SD) · `age_coef` + `age_intercept` |
| `sex_coef`, `sex_intercept` | logistic coefficients and intercept: logit of male sex, computed the same way |
| `ridge_alpha`, `logistic_C` | the chosen penalties |
| `alphas`, `val_mae` | the ridge penalties tried and their validation MAE |
| `Cs`, `val_auc` | the inverse logistic penalties tried and their validation AUC |
| `n_sex_fit` | number of ECGs the logistic regression was fitted to |

In [ ]:
import json

RIDGE_ALPHAS = [1e-1, 1e0, 1e1, 1e2, 1e3, 1e4, 1e5]
LOGISTIC_CS = [1e-3, 1e-2, 1e-1, 1.0]
SEX_FIT_ROWS = 200_000
SEX_FIT_SEED = 0


def fit_probe(key, emb_path, out_dir, record_path):
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    t0 = time.time()
    E = np.load(emb_path, mmap_mode="r")
    lab = model_labels("mimic")
    lab = lab[lab.index < E.shape[0]]
    idx = {s: np.sort(lab.index[lab["split"] == s].to_numpy()) for s in ("train", "val")}
    X = {s: np.asarray(E[v], dtype=np.float32) for s, v in idx.items()}
    mu, sd = X["train"].mean(0), X["train"].std(0) + 1e-6
    for v in X.values():                            # standardise in place: (x - mu) / sd
        v -= mu
        v /= sd
    age = {s: lab.loc[v, "age"].to_numpy() for s, v in idx.items()}
    male = {s: lab.loc[v, "male"].to_numpy() for s, v in idx.items()}
    pid = {s: lab.loc[v, "pid"].to_numpy() for s, v in idx.items()}

    # ridge regression: one Gram matrix of the centred problem (float64, in chunks), solved for every penalty
    ok = ~np.isnan(age["train"])
    rows_ok, yt = np.nonzero(ok)[0], age["train"][ok]
    ym = yt.mean()
    G = np.zeros((X["train"].shape[1],) * 2)
    b = np.zeros(X["train"].shape[1])
    for s in range(0, len(rows_ok), 50_000):
        c = X["train"][rows_ok[s:s + 50_000]].astype(np.float64)
        G += c.T @ c
        b += c.T @ (yt[s:s + 50_000] - ym)
    okv = ~np.isnan(age["val"])
    wv = patient_weights(pid["val"][okv])
    best, val_mae = None, []
    for al in RIDGE_ALPHAS:
        w = np.linalg.solve(G + al * np.eye(len(G)), b)
        mae = float(np.average(np.abs(X["val"][okv] @ w + ym - age["val"][okv]), weights=wv))
        val_mae.append(mae)
        if best is None or mae < best[0]:
            best = (mae, al, w)
    _, alpha, w = best

    # logistic regression for sex on a random subsample of the training split
    rng = np.random.default_rng(SEX_FIT_SEED)
    oks = np.nonzero(~np.isnan(male["train"]))[0]
    sub = rng.choice(oks, min(SEX_FIT_ROWS, len(oks)), replace=False)
    okv_s = ~np.isnan(male["val"])
    bestc, val_auc = None, []
    for C in LOGISTIC_CS:
        m = LogisticRegression(C=C, max_iter=500).fit(X["train"][sub], male["train"][sub])
        auc = float(roc_auc_score(male["val"][okv_s], m.decision_function(X["val"][okv_s])))
        val_auc.append(auc)
        if bestc is None or auc > bestc[0]:
            bestc = (auc, C, m)
    _, C, clf = bestc

    (out_dir / key).mkdir(parents=True, exist_ok=True)
    probe = dict(feature_mean=mu, feature_sd=sd, age_coef=w, age_intercept=np.float64(ym), ridge_alpha=np.float64(alpha),
                 sex_coef=clf.coef_.ravel(), sex_intercept=clf.intercept_[0], logistic_C=np.float64(C),
                 alphas=np.array(RIDGE_ALPHAS), val_mae=np.array(val_mae), Cs=np.array(LOGISTIC_CS), val_auc=np.array(val_auc),
                 n_sex_fit=np.int64(len(sub)))
    np.savez(out_dir / key / "ridge.npz", **probe)
    pa, ps = probe_predict(np.asarray(E[idx["val"]]), probe)
    rec = {"name": key, "ridge_alpha": alpha, "logistic_C": C, "n_sex_fit": int(len(sub)),
           "ridge_val_mae": dict(zip(map(str, RIDGE_ALPHAS), val_mae)), "logistic_val_auc": dict(zip(map(str, LOGISTIC_CS), val_auc)),
           "mimic_val": age_metrics(age["val"], pa, pid["val"], male["val"], ps),
           "n_train": int(len(idx["train"])), "n_val": int(len(idx["val"])), "minutes": round((time.time() - t0) / 60, 1)}
    if QUICK_TEST:
        rec["quick_test"] = True
    record_path.write_text(json.dumps(rec, indent=2), encoding="utf-8")
    print(f"{key}: ridge penalty {alpha:g} (validation MAE {best[0]:.3f} y), logistic C {C:g} (validation AUC {bestc[0]:.4f}); "
          f"validation MAE at the chosen penalty {rec['mimic_val'].get('mae', float('nan')):.2f} y; {rec['minutes']} min")


for key in RUN_PROBES:                              # 1-5 min each on the CPU
    rec = TRAINING_DIR / f"{key}.json"
    if rec.exists() and (TRAINED_DIR / key / "ridge.npz").exists():
        print(f"{key}: exists, skipped")
        continue
    fit_probe(key, EMB_DIR / f"{model_parts(key)[0]}_f16.npy", TRAINED_DIR, rec)

## 4. Fine-tuned versions

The whole network with the two heads is trained, from the pretrained weights, on the training split of MIMIC-IV-ECG or of CODE-15%
(ages of 18–88 years). Settings per foundation model (Supplementary Table 1):

| | ECGFounder | ECG-FM | HuBERT-ECG |
|---|---|---|---|
| ECGs per batch × accumulated batches | 128 × 1 | 32 × 4 | 32 × 4 |
| learning rate, network / heads | 10⁻⁴ / 10⁻³ | 5 × 10⁻⁵ / 10⁻³ | 3 × 10⁻⁵ / 10⁻³ |
| weight decay | 10⁻⁴ | 10⁻² | 10⁻² |

* **Loss**: Huber loss (δ = 1) on (age − 60) / 10, i.e. on age in decades, plus the binary cross-entropy of the sex logit (weight 1);
  each ECG counts once; constant (flat) records are left out of both terms.
* **Optimiser**: AdamW with the learning rates above for the network and for the heads, both heads starting at zero; gradient norm
  clipped at 1; forward pass in bfloat16 mixed precision.
* **Learning rate**: linear warm-up over min(1,000, ⌊T/20⌋ + 1) steps, where T = 8 × ⌈N / (batch × accumulation)⌉ and N is the
  number of training ECGs; then constant on MIMIC-IV-ECG and a cosine decay to 0 at step T on CODE-15%.
* **Validation and checkpoint**: the MAE on the validation split of the same dataset (each patient weighted equally) is computed
  every 500 optimisation steps and at the end of each epoch; the checkpoint with the lowest validation MAE is kept (`best.pt`), and
  training stops after 7 validations without improvement or after 8 epochs.
* **Batches**: records are read in blocks of consecutive rows (2,048 for ECGFounder and ECG-FM, 4,096 for HuBERT-ECG); each epoch
  visits the blocks in a random order, two at a time, and shuffles the records of each pair (generator seeded by the epoch number).
  Batches left at the end of an epoch that do not fill a group of accumulated batches are not used for a step.

The training record `TRAINING/{key}.json` holds `best_epoch` (0-based) and `best_step` of the kept checkpoint, `total_steps`,
`steps_per_epoch` (⌈N / (batch × accumulation)⌉), `cfg`, `early_stop` (True when training ended by the 7-validation rule),
`best_val` (validation results of the kept checkpoint) and `validations` (one entry per validation: step, epoch, validation MAE,
correlation, sex AUC, and whether it was the best so far). A run that is interrupted resumes from `last.pt`, written at the end of
every epoch.

In [ ]:
import math

EPOCHS = 8
EVAL_EVERY = 500
PATIENCE_EVALS = 7
SCHEDULE = {"mimic": "constant", "code15": "cosine"}
SEX_WEIGHT = 1.0
TRAIN_SEED = 0
QUICK_EPOCHS = 3                                    # quick test only
QUICK_EVAL_EVERY = 5


def finetune(key, trained_dir, record_path, quick=False):
    import torch.nn.functional as F
    fm, data = model_parts(key)
    cfg = dict(FINETUNE[fm], epochs=QUICK_EPOCHS if quick else EPOCHS, sched=SCHEDULE[data], patience_evals=PATIENCE_EVALS,
               eval_every=QUICK_EVAL_EVERY if quick else EVAL_EVERY)
    ck_dir = trained_dir / key
    ck_dir.mkdir(parents=True, exist_ok=True)
    dev = gpu_device()
    torch.manual_seed(TRAIN_SEED)
    np.random.seed(TRAIN_SEED)
    gpu_precision(tf32_matmul=True, tf32_conv=True, benchmark=True)

    arr = model_inputs(data, fm)
    lab = model_labels(data)
    lab = lab[lab.index < arr.shape[0]]
    split = {s: np.sort(lab.index[lab["split"] == s].to_numpy()) for s in ("train", "val")}
    if quick:
        split = {"train": split["train"][:QUICK_TRAIN_ROWS], "val": split["val"][:QUICK_VAL_ROWS]}
    age = torch.full((arr.shape[0],), float("nan"))     # targets by row: (age - 60) / 10 and male (NaN = unknown)
    male = torch.full((arr.shape[0],), float("nan"))
    age[lab.index.to_numpy()] = torch.from_numpy(((lab["age"].to_numpy() - AGE_CENTRE) / AGE_SCALE).astype(np.float32))
    male[lab.index.to_numpy()] = torch.from_numpy(lab["male"].to_numpy().astype(np.float32))
    age, male = age.to(dev), male.to(dev)

    model = age_sex_model(fm, pretrained=True).to(dev)
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": cfg["lr_bb"]},
                             {"params": model.head_params(), "lr": cfg["lr_head"]}], weight_decay=cfg["wd"])
    steps = (len(split["train"]) + cfg["bs"] * cfg["accum"] - 1) // (cfg["bs"] * cfg["accum"])   # steps per epoch: ceil(N / (bs x accum))
    total = cfg["epochs"] * steps
    warm = max(1, min(1000, total // 20 + 1))
    if cfg["sched"] == "constant":
        def lr_factor(s):
            return (s + 1) / warm if s < warm else 1.0
    else:
        def lr_factor(s):
            return (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
    n_par = sum(p.numel() for p in model.parameters())
    print(f"{key}: {n_par / 1e6:.1f} million parameters; training {masked(len(split['train']))}, validation "
          f"{masked(len(split['val']))} records; {steps:,} steps per epoch; warm-up {warm} steps; {cfg}", flush=True)

    state = dict(epoch=0, best=None, bad=0, gstep=0, history=[], minutes=0.0, early_stop=False)
    last = ck_dir / "last.pt"
    if last.exists():                                # resume after an interruption
        ck = load_checkpoint(last)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        state = ck["state"]
        print(f"  resumed at epoch {state['epoch'] + 1}, step {state['gstep']}, best validation MAE {state['best']:.3f}")

    def validate(ep, k, epoch_end):
        rows, pa, ps, fl = predict_records(model, arr, split["val"], cfg["bs"], cfg["block"], dev)
        rows, pa, ps = rows[~fl], pa[~fl], ps[~fl]
        m = age_metrics(lab.loc[rows, "age"], pa, lab.loc[rows, "pid"], lab.loc[rows, "male"], ps)
        improved = state["best"] is None or m["mae"] < state["best"]
        if improved:
            state["best"], state["bad"] = m["mae"], 0
            torch.save({"model": model.state_dict(), "epoch": ep, "step": state["gstep"], "val": m}, ck_dir / "best.pt")
        else:
            state["bad"] += 1
        state["history"].append({"step": state["gstep"], "epoch": ep, "step_in_epoch": k, "epoch_end": epoch_end,
                                 "val_mae": m["mae"], "val_r": m.get("r"), "val_sex_auc": m.get("sex_auc"), "best": improved})
        print(f"  step {state['gstep']:,} (epoch {ep + 1}, {k:,}/{steps:,}{', end' if epoch_end else ''}): validation MAE "
              f"{m['mae']:.3f} y, r {m.get('r', float('nan')):.3f}, sex AUC {m.get('sex_auc', float('nan')):.3f}"
              f"{' (best)' if improved else ''}", flush=True)

    patience = cfg["patience_evals"]
    done = state["epoch"] >= cfg["epochs"] or state["bad"] >= patience
    for ep in range(state["epoch"], cfg["epochs"]):
        if done:
            break
        t0, k, nb, run_a, run_s = time.time(), 0, 0, 0.0, 0.0
        model.train()
        opt.zero_grad(set_to_none=True)
        for rows, x in block_batches(arr, split["train"], cfg["bs"], shuffle=True, block=cfg["block"], seed=ep):
            r = torch.from_numpy(rows).to(dev)
            raw = torch.from_numpy(np.ascontiguousarray(x)).to(dev, non_blocking=True)
            xt = model.prep(raw)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                pa, ps, _ = model(xt)
            ya, ys = age[r], male[r]
            ok = ~is_flat(raw)
            ma, ms = ~torch.isnan(ya) & ok, ~torch.isnan(ys) & ok
            la = F.huber_loss(pa.float()[ma], ya[ma], delta=1.0) if ma.any() else pa.sum() * 0
            ls = F.binary_cross_entropy_with_logits(ps.float()[ms], ys[ms]) if ms.any() else ps.sum() * 0
            ((la + SEX_WEIGHT * ls) / cfg["accum"]).backward()
            run_a, run_s, nb = run_a + la.item(), run_s + ls.item(), nb + 1
            if nb % cfg["accum"] == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)
                k += 1
                state["gstep"] += 1
                if k % 500 == 0 or k == 1:
                    print(f"  epoch {ep + 1} step {k:,}/{steps:,}: Huber {run_a / nb:.4f}, cross-entropy {run_s / nb:.4f} "
                          f"({nb * cfg['bs'] / (time.time() - t0):.0f} ECGs per s)", flush=True)
                if state["gstep"] % cfg["eval_every"] == 0 and k < steps:
                    validate(ep, k, False)
                    if state["bad"] >= patience:
                        done = state["early_stop"] = True
                        break
                if k >= steps:
                    break
        if done:
            print(f"  early stop at step {state['gstep']:,} (epoch {ep + 1})")
            break
        validate(ep, k, True)
        state["epoch"] = ep + 1
        state["minutes"] += (time.time() - t0) / 60
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "state": state}, last)
        print(f"  epoch {ep + 1} done in {(time.time() - t0) / 60:.1f} min, step {state['gstep']:,}; learning rate of the network "
              f"{opt.param_groups[0]['lr']:.2e}", flush=True)
        if state["bad"] >= patience:
            state["early_stop"] = True
            print(f"  early stop at the end of epoch {ep + 1}")
            break

    ck = load_checkpoint(ck_dir / "best.pt")
    rec = {"name": key, "best_epoch": int(ck["epoch"]), "best_step": int(ck["step"]), "total_steps": int(state["gstep"]),
           "steps_per_epoch": int(steps), "cfg": cfg, "early_stop": bool(state["early_stop"]), "best_val": ck["val"],
           "validations": state["history"], "n_train": int(len(split["train"])), "n_val": int(len(split["val"])),
           "n_parameters": int(n_par), "train_minutes": round(state["minutes"], 1)}
    if quick:
        rec["quick_test"] = True
    record_path.write_text(json.dumps(rec, indent=2), encoding="utf-8")
    print(f"{key}: kept step {rec['best_step']:,} (epoch {rec['best_epoch'] + 1}), validation MAE {ck['val']['mae']:.3f} y; training "
          f"ended at step {rec['total_steps']:,}{' (early stop)' if rec['early_stop'] else ''}")
    del model, opt
    torch.cuda.empty_cache()


for key in RUN_FINETUNE:                            # 20-110 min per model on one RTX 5090 (see the introduction)
    rec = TRAINING_DIR / f"{key}.json"
    if rec.exists():
        print(f"{key}: trained, skipped")
        continue
    finetune(key, TRAINED_DIR, rec, quick=QUICK_TEST)

## 5. Summary of the adaptation

The selected setting of each model (the last column of Supplementary Table 1).

In [ ]:
rows = []
for key in MODEL_KEYS[1:]:
    p = TRAINING_DIR / f"{key}.json"
    if not p.exists():
        continue
    r = json.loads(p.read_text(encoding="utf-8"))
    if "ridge_alpha" in r:
        rows.append((model_label(key), f"ridge penalty {r['ridge_alpha']:g}, logistic C {r['logistic_C']:g}", ""))
    else:
        rows.append((model_label(key), f"step {r['best_step']:,} (epoch {r['best_epoch'] + 1})",
                     f"ended at step {r['total_steps']:,} ({'early stop' if r['early_stop'] else 'all epochs'})"))
display(pd.DataFrame(rows, columns=["model", "selected setting", "training"]))